# Recursion & Backtracking

Notes and runnable examples on recursion — a function defined in terms of itself — and **backtracking**, the search pattern built on it. The Python-specific angle: CPython's **call stack is finite**, there's **no tail-call optimization**, and naive recursion can explode exponentially — so we'll see when recursion is the right tool, how memoization rescues it, and how to convert it away when depth is the problem.

**Contents**

1. **What recursion is** — base case, recursive case, the call stack
2. **The CPython reality** — the recursion limit & no tail-call optimization
3. **Where it shines (and explodes)** — recursive structure vs exponential blowup
4. **Memoization** — trading space for time (`functools.cache`)
5. **Backtracking** — choose, explore, un-choose
6. **Backtracking in practice** — Sudoku solver & word search
7. **Recursion vs iteration**
8. **Takeaways & cheat-sheet**

## 1. What recursion is

A **recursive** function solves a problem by calling itself on a smaller version of it. Two parts are mandatory:

- a **base case** that stops the recursion (no self-call), and
- a **recursive case** that makes progress *toward* the base case.

Every call gets its own **stack frame** holding its local variables and where to return to. The frames pile up as you recurse deeper, then unwind as each call returns — which you can watch directly:

In [1]:
def factorial(n):
    if n <= 1:                       # base case
        return 1
    return n * factorial(n - 1)      # recursive case (progresses toward 1)

print("factorial(5):", factorial(5))

# the call stack made visible - note how calls nest in, then unwind out:
def countdown(n, depth=0):
    print("  " * depth + f"-> enter countdown({n})")
    if n == 0:
        print("  " * depth + "   base case")
    else:
        countdown(n - 1, depth + 1)
    print("  " * depth + f"<- leave countdown({n})")

countdown(3)


factorial(5): 120
-> enter countdown(3)
  -> enter countdown(2)
    -> enter countdown(1)
      -> enter countdown(0)
         base case
      <- leave countdown(0)
    <- leave countdown(1)
  <- leave countdown(2)
<- leave countdown(3)


## 2. The CPython reality — the recursion limit & no tail-call optimization

The call stack isn't free or infinite. CPython caps recursion depth and raises a catchable `RecursionError` rather than letting you overflow the real C stack and segfault the interpreter. The plain-CPython default is **1000**, though environments can raise it — **Jupyter/IPython sets 3000**, which is what the cell below prints.

Crucially, CPython has **no tail-call optimization (TCO)**. In some languages, when the recursive call is the *last* thing a function does, the compiler reuses the current frame (O(1) stack). Python deliberately doesn't — partly to keep every call visible in tracebacks. So even a perfectly tail-recursive function blows the stack:

In [2]:
import sys
print("recursion limit:", sys.getrecursionlimit())

# A tail-recursive sum: the recursive call is the LAST thing it does.
# TCO languages would reuse the frame; CPython keeps one per call -> overflow.
def tail_sum(n, acc=0):
    if n == 0:
        return acc
    return tail_sum(n - 1, acc + n)

try:
    tail_sum(100_000)
except RecursionError:
    print("tail_sum(100_000) -> RecursionError (no tail-call optimization)")

# the cure is almost always a plain loop:
def iterative_sum(n):
    acc = 0
    while n:
        acc += n
        n -= 1
    return acc

print("iterative_sum(100_000) =", iterative_sum(100_000))


recursion limit: 3000
tail_sum(100_000) -> RecursionError (no tail-call optimization)
iterative_sum(100_000) = 5000050000


You *can* raise the cap with `sys.setrecursionlimit(N)`, but that's a loaded gun: set it too high and a deep recursion overflows the C stack and **crashes** the interpreter instead of raising a clean error. The real lesson: in Python, **deep recursion is a code smell** — convert it to a loop, or to an explicit stack (a `list`) that holds the work the call stack otherwise would. Tail recursion in particular always rewrites to a trivial loop.

## 3. Where it shines (and explodes)

Recursion is the natural fit when the *data itself* is recursive — trees, nested structures, divide-and-conquer (merge sort, the BST traversals from the trees notebook). The code mirrors the structure and reads cleanly.

The trap is **overlapping subproblems**. Naive recursive Fibonacci re-derives the same values over and over, branching into two calls each time — so its cost is **exponential** (~1.6ⁿ):

In [3]:
calls = 0
def fib(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(30) =", fib(30), "computed in", calls, "calls")
print("...the same subproblems are recomputed exponentially many times")


fib(30) = 832040 computed in 2692537 calls
...the same subproblems are recomputed exponentially many times


## 4. Memoization — trading space for time

If a function is **pure** (same inputs ⇒ same output), cache each result so it's computed once. **Memoization** turns that exponential call tree into a linear walk. `functools.cache` does it in one decorator — it stores results in a `dict` keyed by the (hashable) arguments. `functools.lru_cache(maxsize=N)` adds size-bounded eviction using the *circular doubly linked list* we met in the linked-lists notebook.

In [4]:
from functools import cache

@cache                               # = lru_cache(maxsize=None); a dict keyed by the args
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print("fib(100) =", fib(100))        # instant, despite the 21-digit result
print("cache    :", fib.cache_info())  # ~n misses (computed once), the rest hits


fib(100) = 354224848179261915075
cache    : CacheInfo(hits=98, misses=101, maxsize=None, currsize=101)


Each subproblem is now solved exactly once: `fib(100)` does ~100 real computations instead of more calls than there are atoms in the galaxy. Spotting **overlapping subproblems** like this — and caching them — is precisely the idea behind **dynamic programming** (the next notebook), which just turns this top-down memoization into a bottom-up table.

## 5. Backtracking — choose, explore, un-choose

Backtracking is systematic recursion for **search** problems: build a partial solution one choice at a time, recurse, and when a branch is finished or invalid, **undo the last choice** and try the next. It walks the whole decision tree but prunes dead branches early. The skeleton never changes:

```
for each candidate:
    choose it          # extend the partial solution
    explore (recurse)  # solve the rest
    un-choose it       # restore state, try the next candidate
```

Generating all **permutations** and all **subsets** are the two canonical templates:

In [5]:
def permutations(items):
    result, used, path = [], [False] * len(items), []
    def backtrack():
        if len(path) == len(items):
            result.append(path[:])               # a complete arrangement
            return
        for i in range(len(items)):
            if used[i]:
                continue
            used[i] = True; path.append(items[i])    # choose
            backtrack()                              # explore
            path.pop(); used[i] = False              # un-choose
    backtrack()
    return result

def subsets(items):
    result, path = [], []
    def backtrack(start):
        result.append(path[:])                   # every node of the tree is a subset
        for i in range(start, len(items)):
            path.append(items[i])                # choose
            backtrack(i + 1)                      # explore (only forward -> no duplicates)
            path.pop()                            # un-choose
    backtrack(0)
    return result

print("permutations([1,2,3]):", permutations([1, 2, 3]))
print("subsets([1,2,3])     :", subsets([1, 2, 3]))


permutations([1,2,3]): [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
subsets([1,2,3])     : [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]


**N-queens** shows the *pruning* that makes backtracking efficient. Place one queen per row; before recursing, skip any column or diagonal already under attack — abandoning entire subtrees instead of building doomed boards. We track the threatened columns and both diagonals in `set`s for O(1) checks:

In [6]:
def count_nqueens(n):
    cols, diag, anti = set(), set(), set()       # threatened columns / both diagonals
    def place(row):
        if row == n:
            return 1                             # filled every row -> a valid board
        solutions = 0
        for col in range(n):
            if col in cols or (row - col) in diag or (row + col) in anti:
                continue                         # pruned: this square is attacked
            cols.add(col); diag.add(row - col); anti.add(row + col)             # choose
            solutions += place(row + 1)                                          # explore
            cols.discard(col); diag.discard(row - col); anti.discard(row + col)  # un-choose
        return solutions
    return place(0)

for n in (4, 6, 8):
    print(f"  {n}-queens: {count_nqueens(n)} solutions")


  4-queens: 2 solutions
  6-queens: 4 solutions
  8-queens: 92 solutions


## 6. Backtracking in practice — Sudoku & word search

The choose / explore / un-choose skeleton from the previous section scales straight to real puzzles. Two classics show the two flavours of state-restoration: a **Sudoku solver** that tracks constraints implicitly by reading the board back, and a **word search** that mutates the grid in place and must un-mark on backtrack.

**Sudoku** is backtracking over the empty cells. Walk to the first blank, try each digit `1..9`, and keep only those that don't already appear in the cell's row, column, or 3×3 box. *Choose* the digit, *explore* the rest of the board, and *un-choose* (write the blank back) if the recursion dead-ends. The constraint check is what prunes — most digits are rejected before we ever recurse, so the tree stays small:

In [7]:
def solve_sudoku(board):
    """In-place backtracking solver; board is a 9x9 grid, 0 = blank."""
    def is_valid(r, c, d):
        br, bc = 3 * (r // 3), 3 * (c // 3)              # top-left of this 3x3 box
        for i in range(9):
            if board[r][i] == d or board[i][c] == d:     # row / column clash
                return False
            if board[br + i // 3][bc + i % 3] == d:      # box clash
                return False
        return True

    def backtrack():
        for r in range(9):
            for c in range(9):
                if board[r][c] != 0:
                    continue                             # already filled
                for d in range(1, 10):
                    if is_valid(r, c, d):
                        board[r][c] = d                  # choose
                        if backtrack():                  # explore
                            return True
                        board[r][c] = 0                  # un-choose
                return False                             # no digit fits -> dead end
        return True                                      # no blanks left -> solved

    backtrack()
    return board


puzzle = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]
known_solution = [
    [5, 3, 4, 6, 7, 8, 9, 1, 2],
    [6, 7, 2, 1, 9, 5, 3, 4, 8],
    [1, 9, 8, 3, 4, 2, 5, 6, 7],
    [8, 5, 9, 7, 6, 1, 4, 2, 3],
    [4, 2, 6, 8, 5, 3, 7, 9, 1],
    [7, 1, 3, 9, 2, 4, 8, 5, 6],
    [9, 6, 1, 5, 3, 7, 2, 8, 4],
    [2, 8, 7, 4, 1, 9, 6, 3, 5],
    [3, 4, 5, 2, 8, 6, 1, 7, 9],
]

solved = solve_sudoku([row[:] for row in puzzle])

# every row, column, and 3x3 box must be a permutation of 1..9
full = set(range(1, 10))
for r in range(9):
    assert set(solved[r]) == full, f"row {r} is not a permutation of 1..9"
for c in range(9):
    assert {solved[r][c] for r in range(9)} == full, f"col {c} is not a permutation"
for br in (0, 3, 6):
    for bc in (0, 3, 6):
        box = {solved[br + i][bc + j] for i in range(3) for j in range(3)}
        assert box == full, f"box at ({br},{bc}) is not a permutation"
assert solved == known_solution, "solver did not reach the known solution"

print("solved:")
for row in solved:
    print("  " + " ".join(map(str, row)))
print("\nall rows / cols / boxes are permutations of 1..9, and match the known answer")


solved:
  5 3 4 6 7 8 9 1 2
  6 7 2 1 9 5 3 4 8
  1 9 8 3 4 2 5 6 7
  8 5 9 7 6 1 4 2 3
  4 2 6 8 5 3 7 9 1
  7 1 3 9 2 4 8 5 6
  9 6 1 5 3 7 2 8 4
  2 8 7 4 1 9 6 3 5
  3 4 5 2 8 6 1 7 9

all rows / cols / boxes are permutations of 1..9, and match the known answer


**Word search** asks whether a word can be traced through a grid, stepping to the 4 orthogonal neighbours without reusing a cell. From every starting cell we run a DFS that matches one letter at a time. The state we must restore is the *visited* mark: we overwrite a matched cell, recurse, and on the way back **un-mark it** so other paths can use it. (This is the same 4-directional walk as the **grid & matrix traversal** notebook — here it carries a backtracking un-mark.)

In [8]:
def word_search(grid, word):
    rows, cols = len(grid), len(grid[0])

    def dfs(r, c, k):
        if k == len(word):
            return True                          # matched every letter
        if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != word[k]:
            return False                         # off-grid, used, or wrong letter
        saved, grid[r][c] = grid[r][c], "#"      # choose: mark visited
        found = (dfs(r + 1, c, k + 1) or dfs(r - 1, c, k + 1)
                 or dfs(r, c + 1, k + 1) or dfs(r, c - 1, k + 1))   # explore neighbours
        grid[r][c] = saved                       # un-choose: restore the cell
        return found

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))


board = [
    list("ABCE"),
    list("SFCS"),
    list("ADEE"),
]

present = ["ABCCED", "SEE", "SFDA"]    # SFDA snakes around and reuses no cell
absent = ["ABCB", "ABCF", "ZZZ"]       # ABCB would reuse the only 'B'; ZZZ isn't there

for w in present:
    assert word_search([row[:] for row in board], w), f"should find {w!r}"
for w in absent:
    assert not word_search([row[:] for row in board], w), f"should reject {w!r}"

print("found  :", present)
print("rejected:", absent)
print("grid is unmodified after each search (un-mark on backtrack)")


found  : ['ABCCED', 'SEE', 'SFDA']
rejected: ['ABCB', 'ABCF', 'ZZZ']
grid is unmodified after each search (un-mark on backtrack)


## 7. Recursion vs iteration

| Lean recursive when… | Lean iterative when… |
|---|---|
| the data is recursive (trees, nesting, divide & conquer) | it's a simple linear/loop process |
| recursion makes the code dramatically clearer | depth can grow large (CPython will hit its limit) |
| depth stays shallow (≪ 1000) | you're in a hot path (per-call overhead matters) |

Any recursion converts to iteration with an **explicit stack** — a `list` you push/pop to hold the work the call stack would otherwise carry — which is how you process a million-deep structure without a `RecursionError`. Tail recursion converts to a plain loop. And the moment you see **overlapping subproblems**, reach for memoization (or a DP table).

## 8. Takeaways & cheat-sheet

| Concept | Key fact |
|---|---|
| Call stack | each call costs a frame; default depth limit 1000 in CPython (3000 under Jupyter) — see `sys.getrecursionlimit` |
| No tail-call optimization | tail recursion still grows the stack — convert it to a loop |
| `sys.setrecursionlimit` | raises the cap, but set too high → C-stack **segfault**, not a clean error |
| Naive recursion | can be **exponential** when subproblems overlap (e.g. Fibonacci) |
| Memoization (`functools.cache`) | cache pure results → exponential collapses to linear |
| Backtracking | choose / explore / un-choose; prune invalid branches early |
| Recursion → iteration | use an explicit `list` stack to dodge the depth limit |